In [1]:
!pip install openpyxl


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install numpy pandas


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import numpy as np 
import pandas as pd

In [4]:
# Load datasets
customers_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='customers') # Add the path

orders_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='orders') # Add the path 

trans = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='transactions') 

In [5]:
orders_df['order_purchase_date'] = pd.to_datetime(orders_df['order_purchase_date'])

In [6]:
latest_date=orders_df['order_purchase_date'].max()
oldest_date=orders_df['order_purchase_date'].min()
print(latest_date,oldest_date)
cutoff_date = latest_date - pd.DateOffset(months=6)



2018-08-30 13:07:00 2016-10-03 22:31:00


In [7]:
#customer id, total spend,no of orders,loyalty tier,discount applicable
orders_recent=orders_df[orders_df['order_purchase_date']>=cutoff_date]


In [8]:
orders_recent['order_id'].nunique()


2039

In [9]:
customer_order=pd.merge(orders_recent, customers_df, on='customer_id')
customer_order_det=pd.merge(customer_order, trans, on='order_id')


In [10]:
customer_order_det['order_id'].nunique()

2016

In [11]:
customer_order_det.head()

,order_id,customer_id,ship_mode,vendor_id,order_status,order_purchase_date,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,...,country,city,state,contact_number,id,product_id,sales_amt,qty,discount,profit_amt
0,CA-2014-103100,AB-10105,First Class,VEN02,delivered,2018-08-15 09:38:00,2018-08-17 03:10,2018-08-17 15:40,2018-08-22 21:13,2018-09-12,...,United States,omaha,nebraska,734-475-9974,4284,OFF-ST-10001580,35.952,3,0.2,3.5952
1,CA-2014-103100,AB-10105,First Class,VEN02,delivered,2018-08-15 09:38:00,2018-08-17 03:10,2018-08-17 15:40,2018-08-22 21:13,2018-09-12,...,United States,omaha,nebraska,734-475-9974,4283,OFF-AR-10001573,6.990,3,0.0,2.0271
2,CA-2014-103317,DM-13525,First Class,VEN01,delivered,2018-08-06 08:44:00,2018-08-06 09:04,2018-08-07 16:11,2018-08-14 22:18,2018-08-21,...,United States,palm coast,florida,+1 582-333-2978,5439,OFF-ST-10003455,46.530,3,0.0,12.0978
3,CA-2014-103317,DM-13525,First Class,VEN01,delivered,2018-08-06 08:44:00,2018-08-06 09:04,2018-08-07 16:11,2018-08-14 22:18,2018-08-21,...,United States,palm coast,florida,+1 582-333-2978,5437,OFF-AR-10001427,11.960,2,0.0,3.1096
4,CA-2014-103317,DM-13525,First Class,VEN01,delivered,2018-08-06 08:44:00,2018-08-06 09:04,2018-08-07 16:11,2018-08-14 22:18,2018-08-21,...,United States,palm coast,florida,+1 582-333-2978,5438,FUR-TA-10004607,517.405,5,0.3,-81.3065


In [12]:
loyalty_report=customer_order_det.groupby(['customer_id','customer_name']).agg(order_count=('order_id','nunique'),total_spending=('sales_amt','sum')).reset_index()
loyalty_report.head()

,customer_id,customer_name,order_count,total_spending
0,AA-10315,Alex Avila,1,2637.518
1,AA-10375,Allen Armold,3,29.320
2,AA-10480,Andrew Allen,2,2235.244
3,AA-10645,Anna Andreadi,2,230.895
4,AB-10015,Aaron Bergman,2,77.144


In [13]:
conditions=[
    (loyalty_report['total_spending'] <500),
    ((loyalty_report['total_spending'] >=500) & (loyalty_report['total_spending'] <=2000)),
    (loyalty_report['total_spending'] >2000)
]
choices=['Silver','Gold','Platinum']
loyalty_report['loyalty_tier']=np.select(conditions,choices,default='no tier')

In [14]:
conditions=[
    ((loyalty_report['loyalty_tier'] =='Silver')&(loyalty_report['order_count'] <10)),
    ((loyalty_report['loyalty_tier'] =='Silver')&(loyalty_report['order_count'] >=10)),
    ((loyalty_report['loyalty_tier'] =='Gold')&(loyalty_report['order_count'] <10)),
    ((loyalty_report['loyalty_tier'] =='Gold')&(loyalty_report['order_count'] >=10)),
    ((loyalty_report['loyalty_tier'] =='Platinum')&(loyalty_report['order_count'] <10)),
    ((loyalty_report['loyalty_tier'] =='Platinum')&(loyalty_report['order_count'] >=10))
    
]
choices=[2,4,6,8,10,15]
loyalty_report['discount_applicable']=np.select(conditions,choices,default=0)

In [15]:
loyalty_report['discount_applicable'].value_counts()

discount_applicable
6     321
2     262
10    146
Name: count, dtype: int64

In [16]:
loyalty_report.head(10)

,customer_id,customer_name,order_count,total_spending,loyalty_tier,discount_applicable
0,AA-10315,Alex Avila,1,2637.518,Platinum,10
1,AA-10375,Allen Armold,3,29.320,Silver,2
2,AA-10480,Andrew Allen,2,2235.244,Platinum,10
3,AA-10645,Anna Andreadi,2,230.895,Silver,2
4,AB-10015,Aaron Bergman,2,77.144,Silver,2
5,AB-10060,Adam Bellavance,4,3412.382,Platinum,10
6,AB-10105,Adrian Barton,6,757.730,Gold,6
7,AB-10150,Aimee Bixby,3,1654.184,Gold,6
8,AB-10165,Alan Barnes,4,1281.110,Gold,6
9,AB-10255,Alejandro Ballentine,2,38.148,Silver,2


In [17]:
orders_recent['order_id'].count()

np.int64(2041)

In [18]:
new_orders_url = "https://cdn.enqurious.com/documents/517cad90-cfb3-48fe-975e-256d942412ac_neworders.csv"
new_orders_df = pd.read_csv(new_orders_url)

In [19]:
new_orders_df.head()

,customer_id,Order_id,original_price
0,AB-10105,CA-2014-103392,55.462
1,DM-13525,CA-2014-103254,66.374
2,RT-13456,CA-2014-103445,588.678
3,JH-10250,CA-2014-103136,224.731
4,AB-10105,CA-2014-103283,578.572


In [20]:
discounted_pricing_report=pd.merge(loyalty_report, new_orders_df, on='customer_id', how='right')[['Order_id','customer_id','customer_name','original_price','discount_applicable']]

In [21]:
discounted_pricing_report 

,Order_id,customer_id,customer_name,original_price,discount_applicable
0,CA-2014-103392,AB-10105,Adrian Barton,55.462,6.0
1,CA-2014-103254,DM-13525,Don Miller,66.374,6.0
2,CA-2014-103445,RT-13456,NaN,588.678,NaN
3,CA-2014-103136,JH-10250,NaN,224.731,NaN
4,CA-2014-103283,AB-10105,Adrian Barton,578.572,6.0
5,CA-2014-103451,DM-13525,Don Miller,265.371,6.0
6,CA-2014-103269,JH-10250,NaN,13.142,NaN
7,CA-2014-103451,DM-13525,Don Miller,524.170,6.0
8,CA-2014-103326,RT-13456,NaN,133.676,NaN
9,CA-2014-103368,DM-13525,Don Miller,240.350,6.0


In [22]:
discounted_pricing_report['discount_applicable']=discounted_pricing_report['discount_applicable'].fillna(2)

In [23]:
discounted_pricing_report['discounted_price']=discounted_pricing_report['original_price']*(1-discounted_pricing_report['discount_applicable']/100)

In [24]:
discounted_pricing_report

,Order_id,customer_id,customer_name,original_price,discount_applicable,discounted_price
0,CA-2014-103392,AB-10105,Adrian Barton,55.462,6.0,52.13428
1,CA-2014-103254,DM-13525,Don Miller,66.374,6.0,62.39156
2,CA-2014-103445,RT-13456,NaN,588.678,2.0,576.90444
3,CA-2014-103136,JH-10250,NaN,224.731,2.0,220.23638
4,CA-2014-103283,AB-10105,Adrian Barton,578.572,6.0,543.85768
5,CA-2014-103451,DM-13525,Don Miller,265.371,6.0,249.44874
6,CA-2014-103269,JH-10250,NaN,13.142,2.0,12.87916
7,CA-2014-103451,DM-13525,Don Miller,524.170,6.0,492.71980
8,CA-2014-103326,RT-13456,NaN,133.676,2.0,131.00248
9,CA-2014-103368,DM-13525,Don Miller,240.350,6.0,225.92900
